In [1]:
# Cell 1: Imports, device, constants

import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from torchvision.datasets import ImageFolder

# Device + shared constants used throughout training
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 10
INPUT_SHAPE = (3, 180, 180)
BATCH_SIZE = 32
SEED = 42
LEARNING_RATE_FC = 1e-3
LEARNING_RATE_CNN = 1e-4
LEARNING_RATE_RMSPROP = 1e-4
DEBUG_EPOCHS = 1

print("Using device:", DEVICE)


Using device: cpu


In [2]:
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from torchvision.datasets import ImageFolder

# Paths
IMAGE_DIR = "Data/images_original"

# Hyperparameters
BATCH_SIZE = 32
SEED = 42

# Image preprocessing required by the coursework
image_transform = transforms.Compose([
    transforms.Resize((180, 180)),
    transforms.ToTensor()
])

# Load spectrogram image dataset
image_dataset = ImageFolder(
    root=IMAGE_DIR,
    transform=image_transform
)

# Check classes
print("Classes:", image_dataset.classes)
print("Class to index:", image_dataset.class_to_idx)
print("Total image samples:", len(image_dataset))

# Train / validation / test split: 70% / 20% / 10%
total_size = len(image_dataset)
train_size = int(0.7 * total_size)
val_size = int(0.2 * total_size)
test_size = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    image_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(SEED)
)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))
print("Test size:", len(test_dataset))

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Check one batch
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Example labels:", labels[:10])


Classes: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
Class to index: {'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4, 'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9}
Total image samples: 999
Train size: 699
Validation size: 199
Test size: 101
Image batch shape: torch.Size([32, 3, 180, 180])
Label batch shape: torch.Size([32])
Example labels: tensor([7, 1, 6, 6, 6, 6, 0, 9, 4, 6])


In [3]:
# Cell 3: Utility functions

def set_seed(seed=SEED):
    """Set random seeds for reproducibility across Python, NumPy, and PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def count_parameters(model):
    """Return number of trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def smoke_test_model(model, input_shape=INPUT_SHAPE, batch_size=4, num_classes=NUM_CLASSES, device=DEVICE):
    """
    Quick shape/device sanity check for a model.
    Ensures output shape is [batch_size, num_classes].
    """
    model = model.to(device)
    model.eval()
    with torch.no_grad():
        x = torch.randn(batch_size, *input_shape).to(device)
        y = model(x)

    assert y.shape == (batch_size, num_classes), (
        f"Smoke test failed: expected {(batch_size, num_classes)}, got {tuple(y.shape)}"
    )
    print(f"Smoke test passed. Output shape: {tuple(y.shape)}")
    print(f"Trainable parameters: {count_parameters(model):,}")


In [4]:
# Cell 4: Training and evaluation functions

criterion = nn.CrossEntropyLoss()
results_records = []


def train_one_epoch(model, loader, optimizer, criterion, device=DEVICE):
    """Train for one epoch and return average training loss."""
    model.train()
    running_loss = 0.0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        total += images.size(0)

    return running_loss / total


def evaluate(model, loader, criterion, device=DEVICE):
    """Evaluate model and return (avg_loss, accuracy_percent)."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def train_model(model, model_name, architecture, optimizer_name, epochs, train_loader, val_loader, criterion, lr=LEARNING_RATE_FC, device=DEVICE):
    """
    Full training loop with validation tracking.
    Returns model loaded with best validation checkpoint and history dictionary.
    """
    set_seed(SEED)
    model = model.to(device)

    if optimizer_name.lower() == "rmsprop":
        optimizer = optim.RMSprop(model.parameters(), lr=lr)
    elif optimizer_name.lower() == "adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val_acc = -1.0
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"[{model_name}] Epoch {epoch}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%"
        )

    model.load_state_dict(best_state)
    return model, history


def test_model(model, loader, criterion, device=DEVICE):
    """Evaluate trained model on test set."""
    test_loss, test_acc = evaluate(model, loader, criterion, device)
    return test_loss, test_acc


def append_result(model_name, architecture, optimizer_name, epochs, history, test_loss, test_acc):
    """Append one run result to shared records list."""
    best_epoch_idx = int(np.argmax(history["val_acc"]))
    results_records.append({
        "model_name": model_name,
        "architecture": architecture,
        "optimizer": optimizer_name,
        "epochs": epochs,
        "final_training_loss": history["train_loss"][-1],
        "final_validation_loss": history["val_loss"][-1],
        "final_validation_accuracy": history["val_acc"][-1],
        "best_validation_accuracy": max(history["val_acc"]),
        "best_epoch": best_epoch_idx + 1,
        "test_loss": test_loss,
        "test_accuracy": test_acc,
    })


In [5]:
# Cell 5: Net1 model definition only

class Net1(nn.Module):
    """Net1: fully connected network with exactly two hidden layers."""
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        # Net1 has many parameters because a 180x180 RGB image is flattened directly.
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(3 * 180 * 180, 512)
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x


In [8]:
# Cell 6: Net1 smoke test and optional 1-epoch debug run

set_seed(SEED)
net1 = Net1()
smoke_test_model(net1)

RUN_DEBUG_NET1 = True  # Set True to run 1 epoch quick debug training
if RUN_DEBUG_NET1:
    set_seed(SEED)
    net1_debug = Net1()
    net1_debug, net1_debug_history = train_model(
        model=net1_debug,
        model_name="Net1",
        architecture="FC(Flatten->512->128->10)",
        optimizer_name="Adam",
        epochs=DEBUG_EPOCHS,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        lr=LEARNING_RATE_FC,
    )
    dbg_test_loss, dbg_test_acc = test_model(net1_debug, test_loader, criterion)
    print(f"Net1 debug test loss: {dbg_test_loss:.4f}, test acc: {dbg_test_acc:.2f}%")
else:
    print("Net1 debug run skipped. Set RUN_DEBUG_NET1=True to execute.")


Smoke test passed. Output shape: (4, 10)
Trainable parameters: 49,833,866
[Net1] Epoch 1/1 | Train Loss: 7.6485 | Val Loss: 2.8138 | Val Acc: 19.10%
Net1 debug test loss: 2.9220, test acc: 12.87%


In [7]:
# Cell 7: Net1 50-epoch and 100-epoch training calls

RUN_FULL_NET1 = False  # Set True when you want full Net1 training
if RUN_FULL_NET1:
    for n_epochs in [50, 100]:
        set_seed(SEED)
        net1_run = Net1()
        net1_run, net1_history = train_model(
            model=net1_run,
            model_name="Net1",
            architecture="FC(Flatten->512->128->10)",
            optimizer_name="Adam",
            epochs=n_epochs,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            lr=LEARNING_RATE_FC,
        )
        net1_test_loss, net1_test_acc = test_model(net1_run, test_loader, criterion)
        append_result("Net1", "FC(Flatten->512->128->10)", "Adam", n_epochs, net1_history, net1_test_loss, net1_test_acc)
        print(f"Net1 ({n_epochs} epochs) test acc: {net1_test_acc:.2f}%")
else:
    print("Net1 full training skipped. Set RUN_FULL_NET1=True to execute.")


Net1 full training skipped. Set RUN_FULL_NET1=True to execute.


In [9]:
# Cell 8: Net2 model definition only

class Net2(nn.Module):
    """
    Net2: Coursework Figure 1 style CNN
    Input -> Conv+ReLU -> Conv+ReLU -> MaxPool
          -> Conv+ReLU -> Conv+ReLU -> MaxPool
          -> FC+ReLU -> FC output
    """
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, *INPUT_SHAPE)
            flattened_dim = self.features(dummy).view(1, -1).size(1)

        self.classifier = nn.Sequential(
            nn.Linear(flattened_dim, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


In [11]:
# Cell 9: Net2 smoke test and optional 1-epoch debug run

set_seed(SEED)
net2 = Net2()
smoke_test_model(net2)

RUN_DEBUG_NET2 = True
if RUN_DEBUG_NET2:
    set_seed(SEED)
    net2_debug = Net2()
    net2_debug, net2_debug_history = train_model(
        model=net2_debug,
        model_name="Net2",
        architecture="CNN(Fig1 4conv+2pool+fc256)",
        optimizer_name="Adam",
        epochs=DEBUG_EPOCHS,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        # Use lower LR for CNNs due to observed instability with larger LR.
        lr=LEARNING_RATE_CNN,
    )
    dbg_test_loss, dbg_test_acc = test_model(net2_debug, test_loader, criterion)
    print(f"Net2 debug test loss: {dbg_test_loss:.4f}, test acc: {dbg_test_acc:.2f}%")
else:
    print("Net2 debug run skipped. Set RUN_DEBUG_NET2=True to execute.")


Smoke test passed. Output shape: (4, 10)
Trainable parameters: 33,245,994
[Net2] Epoch 1/1 | Train Loss: 2.3552 | Val Loss: 2.2996 | Val Acc: 15.58%
Net2 debug test loss: 2.3089, test acc: 5.94%


In [12]:
# Cell 10: Net2 50-epoch and 100-epoch training calls

RUN_FULL_NET2 = False
if RUN_FULL_NET2:
    for n_epochs in [50, 100]:
        set_seed(SEED)
        net2_run = Net2()
        net2_run, net2_history = train_model(
            model=net2_run,
            model_name="Net2",
            architecture="CNN(Fig1 4conv+2pool+fc256)",
            optimizer_name="Adam",
            epochs=n_epochs,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            lr=LEARNING_RATE_CNN,
        )
        net2_test_loss, net2_test_acc = test_model(net2_run, test_loader, criterion)
        append_result("Net2", "CNN(Fig1 4conv+2pool+fc256)", "Adam", n_epochs, net2_history, net2_test_loss, net2_test_acc)
        print(f"Net2 ({n_epochs} epochs) test acc: {net2_test_acc:.2f}%")
else:
    print("Net2 full training skipped. Set RUN_FULL_NET2=True to execute.")


Net2 full training skipped. Set RUN_FULL_NET2=True to execute.


In [13]:
# Cell 11: Net3 model definition only

class Net3(nn.Module):
    """
    Net3: Same architecture as Net2, but with BatchNorm2d
    after each convolution and before ReLU.
    """
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, *INPUT_SHAPE)
            flattened_dim = self.features(dummy).view(1, -1).size(1)

        self.classifier = nn.Sequential(
            nn.Linear(flattened_dim, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


In [14]:
# Cell 12: Net3 smoke test and optional 1-epoch debug run

set_seed(SEED)
net3 = Net3()
smoke_test_model(net3)

RUN_DEBUG_NET3 = True
if RUN_DEBUG_NET3:
    set_seed(SEED)
    net3_debug = Net3()
    net3_debug, net3_debug_history = train_model(
        model=net3_debug,
        model_name="Net3",
        architecture="CNN+BN(Fig1 4conv+2pool+fc256)",
        optimizer_name="Adam",
        epochs=DEBUG_EPOCHS,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        # Use lower LR for CNN+BatchNorm due to unstable early debug losses.
        lr=LEARNING_RATE_CNN,
    )
    dbg_test_loss, dbg_test_acc = test_model(net3_debug, test_loader, criterion)
    print(f"Net3 debug test loss: {dbg_test_loss:.4f}, test acc: {dbg_test_acc:.2f}%")
else:
    print("Net3 debug run skipped. Set RUN_DEBUG_NET3=True to execute.")


Smoke test passed. Output shape: (4, 10)
Trainable parameters: 33,246,378
[Net3] Epoch 1/1 | Train Loss: 24.2430 | Val Loss: 3.7669 | Val Acc: 9.55%
Net3 debug test loss: 4.7888, test acc: 13.86%


In [15]:
# Cell 13: Net3 50-epoch and 100-epoch training calls

RUN_FULL_NET3 = False
if RUN_FULL_NET3:
    for n_epochs in [50, 100]:
        set_seed(SEED)
        net3_run = Net3()
        net3_run, net3_history = train_model(
            model=net3_run,
            model_name="Net3",
            architecture="CNN+BN(Fig1 4conv+2pool+fc256)",
            optimizer_name="Adam",
            epochs=n_epochs,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            lr=LEARNING_RATE_CNN,
        )
        net3_test_loss, net3_test_acc = test_model(net3_run, test_loader, criterion)
        append_result("Net3", "CNN+BN(Fig1 4conv+2pool+fc256)", "Adam", n_epochs, net3_history, net3_test_loss, net3_test_acc)
        print(f"Net3 ({n_epochs} epochs) test acc: {net3_test_acc:.2f}%")
else:
    print("Net3 full training skipped. Set RUN_FULL_NET3=True to execute.")


Net3 full training skipped. Set RUN_FULL_NET3=True to execute.


In [16]:
# Cell 14: Net4 setup using Net3 architecture and RMSProp

# Net4 must use the SAME architecture as Net3.
# We reuse Net3 directly to keep comparison fair.
# RMSprop is used only to satisfy the coursework requirement of same architecture + different optimizer.
Net4 = Net3
NET4_ARCH = "CNN+BN(Fig1 4conv+2pool+fc256)"
NET4_OPTIMIZER = "RMSprop"

print("Net4 setup complete: using Net3 architecture with RMSprop optimizer.")


Net4 setup complete: using Net3 architecture with RMSprop optimizer.


In [17]:
# Cell 15: Net4 smoke test and optional 1-epoch debug run

set_seed(SEED)
net4 = Net4()
smoke_test_model(net4)

RUN_DEBUG_NET4 = True
if RUN_DEBUG_NET4:
    set_seed(SEED)
    net4_debug = Net4()
    net4_debug, net4_debug_history = train_model(
        model=net4_debug,
        model_name="Net4",
        architecture=NET4_ARCH,
        optimizer_name=NET4_OPTIMIZER,
        epochs=DEBUG_EPOCHS,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        # Use lower LR for RMSprop because larger LR showed unstable early debug losses.
        lr=LEARNING_RATE_RMSPROP,
    )
    dbg_test_loss, dbg_test_acc = test_model(net4_debug, test_loader, criterion)
    print(f"Net4 debug test loss: {dbg_test_loss:.4f}, test acc: {dbg_test_acc:.2f}%")
else:
    print("Net4 debug run skipped. Set RUN_DEBUG_NET4=True to execute.")


Smoke test passed. Output shape: (4, 10)
Trainable parameters: 33,246,378
[Net4] Epoch 1/1 | Train Loss: 154.6020 | Val Loss: 4.9564 | Val Acc: 26.13%
Net4 debug test loss: 5.3167, test acc: 20.79%


In [18]:
# Cell 16: Net4 50-epoch and 100-epoch training calls

RUN_FULL_NET4 = False
if RUN_FULL_NET4:
    for n_epochs in [50, 100]:
        set_seed(SEED)
        net4_run = Net4()
        net4_run, net4_history = train_model(
            model=net4_run,
            model_name="Net4",
            architecture=NET4_ARCH,
            optimizer_name=NET4_OPTIMIZER,
            epochs=n_epochs,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            lr=LEARNING_RATE_RMSPROP,
        )
        net4_test_loss, net4_test_acc = test_model(net4_run, test_loader, criterion)
        append_result("Net4", NET4_ARCH, NET4_OPTIMIZER, n_epochs, net4_history, net4_test_loss, net4_test_acc)
        print(f"Net4 ({n_epochs} epochs) test acc: {net4_test_acc:.2f}%")
else:
    print("Net4 full training skipped. Set RUN_FULL_NET4=True to execute.")


Net4 full training skipped. Set RUN_FULL_NET4=True to execute.


In [19]:
# Cell 17: Results table and CSV saving

# Build DataFrame from completed training runs.
results_df = pd.DataFrame(results_records)

if results_df.empty:
    print("No full training runs have been logged yet.")
    print("Run Net1-Net4 full training cells with RUN_FULL_* = True, then rerun this cell.")
else:
    results_df = results_df.sort_values(["model_name", "epochs"]).reset_index(drop=True)
    display(results_df)

# Save CSV even if empty, so coursework file path is always created.
results_df.to_csv("results_image_models.csv", index=False)
print("Saved results to results_image_models.csv")


No full training runs have been logged yet.
Run Net1-Net4 full training cells with RUN_FULL_* = True, then rerun this cell.
Saved results to results_image_models.csv


In [ ]:
# Cell 18: Audio imports and audio constants

import os
import importlib.util
from torch.utils.data import Dataset, TensorDataset

if importlib.util.find_spec("librosa") is not None:
    import librosa
    LIBROSA_AVAILABLE = True
else:
    LIBROSA_AVAILABLE = False
    print("Warning: librosa is not installed. Audio cells requiring feature extraction will not run until librosa is available.")

AUDIO_DIR = "Data/genres_original"
SAMPLE_RATE = 22050
DURATION = 30
N_MFCC = 40
MAX_AUDIO_TIME_STEPS = 1300  # fixed length for MFCC time axis after padding/truncation
AUDIO_BATCH_SIZE = 16
AUDIO_DEBUG_EPOCHS = 1
AUDIO_FULL_EPOCHS = 20
GAN_DEBUG_EPOCHS = 1
GAN_FULL_EPOCHS = 20
AUDIO_LEARNING_RATE = 1e-3
GAN_LEARNING_RATE = 1e-4

print("Audio config ready.")
print(f"LIBROSA_AVAILABLE={LIBROSA_AVAILABLE}, AUDIO_DIR={AUDIO_DIR}")


In [ ]:
# Cell 19: Audio dataset class

class AudioMFCCDataset(Dataset):
    """
    Loads GTZAN-style genre folders and returns fixed-length MFCC sequences.
    Feature output shape: [MAX_AUDIO_TIME_STEPS, N_MFCC].
    """
    def __init__(self, audio_dir=AUDIO_DIR, sample_rate=SAMPLE_RATE, duration=DURATION, n_mfcc=N_MFCC, max_time_steps=MAX_AUDIO_TIME_STEPS):
        self.audio_dir = audio_dir
        self.sample_rate = sample_rate
        self.duration = duration
        self.n_mfcc = n_mfcc
        self.max_time_steps = max_time_steps

        # Keep ImageFolder-compatible class order for reproducibility/reporting.
        self.classes = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]
        self.class_to_idx = {name: idx for idx, name in enumerate(self.classes)}

        self.samples = []
        missing_folders = []
        for genre in self.classes:
            class_dir = os.path.join(self.audio_dir, genre)
            if not os.path.isdir(class_dir):
                missing_folders.append(genre)
                continue

            for root, _, files in os.walk(class_dir):
                for fname in sorted(files):
                    if fname.lower().endswith('.wav'):
                        fpath = os.path.join(root, fname)
                        self.samples.append((fpath, self.class_to_idx[genre]))

        if missing_folders:
            print(f"Warning: missing genre folders: {missing_folders}")

        if len(self.samples) == 0:
            print("Warning: no audio .wav samples found in AUDIO_DIR.")

    def __len__(self):
        return len(self.samples)

    def _extract_mfcc(self, path):
        y, _ = librosa.load(path, sr=self.sample_rate, duration=self.duration)
        mfcc = librosa.feature.mfcc(y=y, sr=self.sample_rate, n_mfcc=self.n_mfcc)
        mfcc = mfcc.T  # [time, n_mfcc]

        if mfcc.shape[0] < self.max_time_steps:
            pad_len = self.max_time_steps - mfcc.shape[0]
            mfcc = np.pad(mfcc, ((0, pad_len), (0, 0)), mode='constant')
        else:
            mfcc = mfcc[:self.max_time_steps, :]

        return mfcc.astype(np.float32)

    def __getitem__(self, idx):
        if not LIBROSA_AVAILABLE:
            raise RuntimeError("librosa is required for AudioMFCCDataset. Please install librosa.")

        path, label = self.samples[idx]
        try:
            mfcc = self._extract_mfcc(path)
        except Exception as exc:
            print(f"Warning: failed to load/extract MFCC from {path}: {exc}")
            mfcc = np.zeros((self.max_time_steps, self.n_mfcc), dtype=np.float32)

        features = torch.tensor(mfcc, dtype=torch.float32)
        label_t = torch.tensor(label, dtype=torch.long)
        return features, label_t


In [ ]:
# Cell 20: Audio dataset loading and PyTorch split

if not LIBROSA_AVAILABLE:
    print("Skipping audio dataset loading because librosa is unavailable.")
    audio_dataset = None
else:
    audio_dataset = AudioMFCCDataset(AUDIO_DIR)

if audio_dataset is not None and len(audio_dataset) > 0:
    print("Audio classes:", audio_dataset.classes)
    print("Total audio samples:", len(audio_dataset))

    sample_x, sample_y = audio_dataset[0]
    print("One feature shape:", tuple(sample_x.shape))
    print("One label:", int(sample_y))

    total_audio = len(audio_dataset)
    train_audio_size = int(0.7 * total_audio)
    val_audio_size = int(0.2 * total_audio)
    test_audio_size = total_audio - train_audio_size - val_audio_size

    train_audio_dataset, val_audio_dataset, test_audio_dataset = random_split(
        audio_dataset,
        [train_audio_size, val_audio_size, test_audio_size],
        generator=torch.Generator().manual_seed(SEED)
    )

    train_audio_loader = DataLoader(train_audio_dataset, batch_size=AUDIO_BATCH_SIZE, shuffle=True)
    val_audio_loader = DataLoader(val_audio_dataset, batch_size=AUDIO_BATCH_SIZE, shuffle=False)
    test_audio_loader = DataLoader(test_audio_dataset, batch_size=AUDIO_BATCH_SIZE, shuffle=False)

    print("Train audio size:", len(train_audio_dataset))
    print("Validation audio size:", len(val_audio_dataset))
    print("Test audio size:", len(test_audio_dataset))

    batch_x, batch_y = next(iter(train_audio_loader))
    print("Audio batch feature shape:", tuple(batch_x.shape))  # [batch, time, n_mfcc]
    print("Audio batch label shape:", tuple(batch_y.shape))
else:
    print("Audio dataset is empty or unavailable. Check AUDIO_DIR and librosa installation.")


In [ ]:
# Cell 21: Net5 LSTM model definition

class Net5LSTM(nn.Module):
    """
    Net5 uses MFCC sequences extracted from original audio samples.
    MFCC features are used because raw waveform LSTM training is computationally expensive.
    """
    def __init__(self, input_size=N_MFCC, hidden_size=128, num_layers=2, num_classes=NUM_CLASSES):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3 if num_layers > 1 else 0.0,
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        output, (h_n, c_n) = self.lstm(x)
        last_hidden = h_n[-1]
        logits = self.classifier(last_hidden)
        return logits


In [ ]:
# Cell 22: Audio training and evaluation functions

audio_criterion = nn.CrossEntropyLoss()
audio_results_records = []


def smoke_test_audio_model(model, seq_len=MAX_AUDIO_TIME_STEPS, n_mfcc=N_MFCC, batch_size=4, num_classes=NUM_CLASSES, device=DEVICE):
    model = model.to(device)
    model.eval()
    with torch.no_grad():
        x = torch.randn(batch_size, seq_len, n_mfcc).to(device)
        y = model(x)
    assert y.shape == (batch_size, num_classes), f"Expected {(batch_size, num_classes)}, got {tuple(y.shape)}"
    print(f"Audio smoke test passed. Output shape: {tuple(y.shape)}")
    print(f"Trainable parameters: {count_parameters(model):,}")


def train_one_epoch_audio(model, loader, optimizer, criterion, device=DEVICE):
    model.train()
    running_loss, total = 0.0, 0
    for features, labels in loader:
        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(features)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * features.size(0)
        total += features.size(0)
    return running_loss / max(total, 1)


def evaluate_audio(model, loader, criterion, device=DEVICE):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for features, labels in loader:
            features = features.to(device)
            labels = labels.to(device)
            logits = model(features)
            loss = criterion(logits, labels)

            running_loss += loss.item() * features.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = running_loss / max(total, 1)
    acc = 100.0 * correct / max(total, 1)
    return avg_loss, acc


def train_audio_model(model, model_name, epochs, train_loader, val_loader, criterion=audio_criterion, lr=AUDIO_LEARNING_RATE, optimizer_name="adam", device=DEVICE):
    set_seed(SEED)
    model = model.to(device)

    if optimizer_name.lower() == "adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    elif optimizer_name.lower() == "rmsprop":
        optimizer = optim.RMSprop(model.parameters(), lr=lr)
    else:
        raise ValueError(f"Unsupported optimizer for audio model: {optimizer_name}")

    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val_acc = -1.0
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch_audio(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate_audio(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"[{model_name}] Epoch {epoch}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%"
        )

    model.load_state_dict(best_state)
    return model, history


def test_audio_model(model, loader, criterion=audio_criterion, device=DEVICE):
    return evaluate_audio(model, loader, criterion, device)


def append_audio_result(model_name, optimizer_name, epochs, history, test_loss, test_acc):
    best_epoch_idx = int(np.argmax(history["val_acc"]))
    audio_results_records.append({
        "model_name": model_name,
        "optimizer": optimizer_name,
        "epochs": epochs,
        "final_training_loss": history["train_loss"][-1],
        "final_validation_loss": history["val_loss"][-1],
        "final_validation_accuracy": history["val_acc"][-1],
        "best_validation_accuracy": max(history["val_acc"]),
        "best_epoch": best_epoch_idx + 1,
        "test_loss": test_loss,
        "test_accuracy": test_acc,
    })


In [ ]:
# Cell 23: Net5 smoke test and 1-epoch debug run

set_seed(SEED)
net5 = Net5LSTM()
smoke_test_audio_model(net5)

RUN_DEBUG_NET5 = True
RUN_FULL_NET5 = False

if RUN_DEBUG_NET5 and LIBROSA_AVAILABLE and 'train_audio_loader' in globals():
    set_seed(SEED)
    net5_debug = Net5LSTM()
    net5_debug, net5_debug_history = train_audio_model(
        model=net5_debug,
        model_name="Net5",
        epochs=AUDIO_DEBUG_EPOCHS,
        train_loader=train_audio_loader,
        val_loader=val_audio_loader,
        lr=AUDIO_LEARNING_RATE,
        optimizer_name="adam",
    )
    net5_dbg_test_loss, net5_dbg_test_acc = test_audio_model(net5_debug, test_audio_loader)
    print(f"Net5 debug test loss: {net5_dbg_test_loss:.4f}, test acc: {net5_dbg_test_acc:.2f}%")
else:
    print("Net5 debug run skipped (check RUN_DEBUG_NET5, librosa, and audio loaders).")


In [ ]:
# Cell 24: Net5 full training cell

RUN_FULL_NET5 = False
if RUN_FULL_NET5 and LIBROSA_AVAILABLE and 'train_audio_loader' in globals():
    set_seed(SEED)
    net5_full = Net5LSTM()
    net5_full, net5_history = train_audio_model(
        model=net5_full,
        model_name="Net5",
        epochs=AUDIO_FULL_EPOCHS,
        train_loader=train_audio_loader,
        val_loader=val_audio_loader,
        lr=AUDIO_LEARNING_RATE,
        optimizer_name="adam",
    )
    net5_test_loss, net5_test_acc = test_audio_model(net5_full, test_audio_loader)
    append_audio_result("Net5", "Adam", AUDIO_FULL_EPOCHS, net5_history, net5_test_loss, net5_test_acc)
    print(f"Net5 full test acc: {net5_test_acc:.2f}%")
else:
    print("Net5 full training skipped. Set RUN_FULL_NET5=True to execute.")


In [ ]:
# Cell 25: Feature-level GAN dataset preparation

# Feature-level GAN approximation: GAN learns MFCC feature vectors, not raw waveforms.
# This keeps compute manageable while still augmenting sequence features for Net6.
if LIBROSA_AVAILABLE and 'train_audio_loader' in globals():
    real_feature_batches = []
    real_label_batches = []

    for features, labels in train_audio_loader:
        flat_features = features.view(features.size(0), -1)
        real_feature_batches.append(flat_features)
        real_label_batches.append(labels)

    real_flat_features = torch.cat(real_feature_batches, dim=0)
    real_train_labels = torch.cat(real_label_batches, dim=0)
    GAN_FEATURE_DIM = real_flat_features.shape[1]
    gan_real_dataset = TensorDataset(real_flat_features)
    gan_real_loader = DataLoader(gan_real_dataset, batch_size=AUDIO_BATCH_SIZE, shuffle=True)

    print("GAN_FEATURE_DIM:", GAN_FEATURE_DIM)
    print("Real MFCC feature vectors for GAN:", real_flat_features.shape[0])
else:
    GAN_FEATURE_DIM = MAX_AUDIO_TIME_STEPS * N_MFCC
    print("Skipping GAN real-data preparation (audio loader not available).")


In [ ]:
# Cell 26: Lightweight feature-level GAN model

NOISE_DIM = 100

class MFCCGenerator(nn.Module):
    def __init__(self, noise_dim=NOISE_DIM, output_dim=None):
        super().__init__()
        output_dim = output_dim if output_dim is not None else GAN_FEATURE_DIM
        self.net = nn.Sequential(
            nn.Linear(noise_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, output_dim),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z)


class MFCCDiscriminator(nn.Module):
    def __init__(self, input_dim=None):
        super().__init__()
        input_dim = input_dim if input_dim is not None else GAN_FEATURE_DIM
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)

set_seed(SEED)
generator = MFCCGenerator().to(DEVICE)
set_seed(SEED)
discriminator = MFCCDiscriminator().to(DEVICE)

print("Feature-level GAN models initialized.")


In [ ]:
# Cell 27: GAN training function

gan_criterion = nn.BCELoss()


def train_gan(generator, discriminator, real_loader, epochs, noise_dim=NOISE_DIM, lr=GAN_LEARNING_RATE, device=DEVICE):
    g_opt = optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
    d_opt = optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))

    history = {"g_loss": [], "d_loss": []}
    generator.train()
    discriminator.train()

    for epoch in range(1, epochs + 1):
        running_g, running_d, n_batches = 0.0, 0.0, 0

        for (real_x,) in real_loader:
            real_x = real_x.to(device)
            bs = real_x.size(0)

            real_targets = torch.ones(bs, 1, device=device)
            fake_targets = torch.zeros(bs, 1, device=device)

            # Train Discriminator
            d_opt.zero_grad()
            real_pred = discriminator(real_x)
            d_real_loss = gan_criterion(real_pred, real_targets)

            z = torch.randn(bs, noise_dim, device=device)
            fake_x = generator(z).detach()
            fake_pred = discriminator(fake_x)
            d_fake_loss = gan_criterion(fake_pred, fake_targets)

            d_loss = d_real_loss + d_fake_loss
            d_loss.backward()
            d_opt.step()

            # Train Generator
            g_opt.zero_grad()
            z = torch.randn(bs, noise_dim, device=device)
            gen_x = generator(z)
            gen_pred = discriminator(gen_x)
            g_loss = gan_criterion(gen_pred, real_targets)
            g_loss.backward()
            g_opt.step()

            running_d += d_loss.item()
            running_g += g_loss.item()
            n_batches += 1

        epoch_d = running_d / max(n_batches, 1)
        epoch_g = running_g / max(n_batches, 1)
        history["d_loss"].append(epoch_d)
        history["g_loss"].append(epoch_g)

        print(f"[GAN] Epoch {epoch}/{epochs} | D Loss: {epoch_d:.4f} | G Loss: {epoch_g:.4f}")

    return history


In [ ]:
# Cell 28: GAN debug run

RUN_DEBUG_GAN = True
RUN_FULL_GAN = False

if RUN_DEBUG_GAN and LIBROSA_AVAILABLE and 'gan_real_loader' in globals():
    set_seed(SEED)
    generator = MFCCGenerator().to(DEVICE)
    set_seed(SEED)
    discriminator = MFCCDiscriminator().to(DEVICE)

    gan_debug_history = train_gan(
        generator=generator,
        discriminator=discriminator,
        real_loader=gan_real_loader,
        epochs=GAN_DEBUG_EPOCHS,
        lr=GAN_LEARNING_RATE,
    )

    with torch.no_grad():
        z = torch.randn(4, NOISE_DIM, device=DEVICE)
        synth_flat = generator(z)
        synth_seq = synth_flat.view(4, MAX_AUDIO_TIME_STEPS, N_MFCC)
        print("Synthetic flattened shape:", tuple(synth_flat.shape))
        print("Synthetic sequence shape:", tuple(synth_seq.shape))
else:
    print("GAN debug run skipped (check RUN_DEBUG_GAN, librosa, and GAN loader).")


In [ ]:
# Cell 29: Create augmented training dataset for Net6

if LIBROSA_AVAILABLE and 'generator' in globals() and 'real_flat_features' in globals():
    generator.eval()

    num_real = real_flat_features.size(0)
    num_synth = num_real

    with torch.no_grad():
        z = torch.randn(num_synth, NOISE_DIM, device=DEVICE)
        synth_flat = generator(z).cpu()

    synth_seq = synth_flat.view(num_synth, MAX_AUDIO_TIME_STEPS, N_MFCC)

    # Simple practical label assignment for unconditional GAN:
    # sample from empirical training label distribution.
    sampled_idx = torch.randint(low=0, high=real_train_labels.size(0), size=(num_synth,))
    synth_labels = real_train_labels[sampled_idx].clone()

    real_seq = real_flat_features.view(num_real, MAX_AUDIO_TIME_STEPS, N_MFCC)
    real_labels = real_train_labels.clone()

    aug_features = torch.cat([real_seq, synth_seq], dim=0)
    aug_labels = torch.cat([real_labels, synth_labels], dim=0)

    augmented_train_dataset = TensorDataset(aug_features, aug_labels)
    augmented_train_audio_loader = DataLoader(augmented_train_dataset, batch_size=AUDIO_BATCH_SIZE, shuffle=True)

    print("Number of real training samples:", num_real)
    print("Number of synthetic samples:", num_synth)
    print("Number of augmented training samples:", len(augmented_train_dataset))
else:
    print("Skipping augmented dataset creation (generator or real features unavailable).")


In [ ]:
# Cell 30: Net6 LSTM model definition/setup

# Net6 uses the SAME LSTM architecture as Net5.
# Difference: Net6 is trained on augmented training data (real + GAN-generated MFCC-like features).
Net6LSTM = Net5LSTM

print("Net6 setup complete: same architecture as Net5, trained with augmented MFCC features.")


In [ ]:
# Cell 31: Net6 smoke test and 1-epoch debug run

set_seed(SEED)
net6 = Net6LSTM()
smoke_test_audio_model(net6)

RUN_DEBUG_NET6 = True
RUN_FULL_NET6 = False

# Validation and test remain real-audio only (not augmented).
if RUN_DEBUG_NET6 and LIBROSA_AVAILABLE and 'augmented_train_audio_loader' in globals():
    set_seed(SEED)
    net6_debug = Net6LSTM()
    net6_debug, net6_debug_history = train_audio_model(
        model=net6_debug,
        model_name="Net6",
        epochs=AUDIO_DEBUG_EPOCHS,
        train_loader=augmented_train_audio_loader,
        val_loader=val_audio_loader,
        lr=AUDIO_LEARNING_RATE,
        optimizer_name="adam",
    )
    net6_dbg_test_loss, net6_dbg_test_acc = test_audio_model(net6_debug, test_audio_loader)
    print(f"Net6 debug test loss: {net6_dbg_test_loss:.4f}, test acc: {net6_dbg_test_acc:.2f}%")
else:
    print("Net6 debug run skipped (check RUN_DEBUG_NET6, librosa, and augmented loader).")


In [ ]:
# Cell 32: Net6 full training cell

RUN_FULL_NET6 = False
if RUN_FULL_NET6 and LIBROSA_AVAILABLE and 'augmented_train_audio_loader' in globals():
    set_seed(SEED)
    net6_full = Net6LSTM()
    net6_full, net6_history = train_audio_model(
        model=net6_full,
        model_name="Net6",
        epochs=AUDIO_FULL_EPOCHS,
        train_loader=augmented_train_audio_loader,
        val_loader=val_audio_loader,
        lr=AUDIO_LEARNING_RATE,
        optimizer_name="adam",
    )
    net6_test_loss, net6_test_acc = test_audio_model(net6_full, test_audio_loader)
    append_audio_result("Net6", "Adam", AUDIO_FULL_EPOCHS, net6_history, net6_test_loss, net6_test_acc)
    print(f"Net6 full test acc: {net6_test_acc:.2f}%")
else:
    print("Net6 full training skipped. Set RUN_FULL_NET6=True to execute.")


In [ ]:
# Cell 33: Audio results table and CSV saving

audio_results_df = pd.DataFrame(audio_results_records)

if audio_results_df.empty:
    print("No Net5/Net6 full-training runs logged yet.")
else:
    audio_results_df = audio_results_df.sort_values(["model_name", "epochs"]).reset_index(drop=True)
    display(audio_results_df)

audio_results_df.to_csv("results_audio_models.csv", index=False)
print("Saved audio results to results_audio_models.csv")


In [ ]:
# Cell 34: Overnight run control cell

RUN_OVERNIGHT_NET5_NET6 = False

if RUN_OVERNIGHT_NET5_NET6:
    print("[Overnight] Starting Net5/Net6 pipeline...")

    if not LIBROSA_AVAILABLE:
        raise RuntimeError("[Overnight] librosa is required for audio pipeline.")

    if 'train_audio_loader' not in globals():
        raise RuntimeError("[Overnight] train_audio_loader is not available. Run Cell 20 first.")

    # 1) Train Net5 full
    print("[Overnight] Step 1/6: Training Net5 full...")
    set_seed(SEED)
    overnight_net5 = Net5LSTM()
    overnight_net5, overnight_net5_hist = train_audio_model(
        model=overnight_net5,
        model_name="Net5",
        epochs=AUDIO_FULL_EPOCHS,
        train_loader=train_audio_loader,
        val_loader=val_audio_loader,
        lr=AUDIO_LEARNING_RATE,
    )
    overnight_net5_test_loss, overnight_net5_test_acc = test_audio_model(overnight_net5, test_audio_loader)
    append_audio_result("Net5_overnight", "Adam", AUDIO_FULL_EPOCHS, overnight_net5_hist, overnight_net5_test_loss, overnight_net5_test_acc)

    # 2) Train GAN full
    print("[Overnight] Step 2/6: Training feature-level GAN full...")
    set_seed(SEED)
    overnight_gen = MFCCGenerator().to(DEVICE)
    set_seed(SEED)
    overnight_disc = MFCCDiscriminator().to(DEVICE)
    _ = train_gan(
        generator=overnight_gen,
        discriminator=overnight_disc,
        real_loader=gan_real_loader,
        epochs=GAN_FULL_EPOCHS,
        lr=GAN_LEARNING_RATE,
    )

    # 3) Generate synthetic MFCC features
    print("[Overnight] Step 3/6: Generating synthetic MFCC features...")
    overnight_gen.eval()
    n_real = real_flat_features.size(0)
    with torch.no_grad():
        z = torch.randn(n_real, NOISE_DIM, device=DEVICE)
        overnight_synth_flat = overnight_gen(z).cpu()
    overnight_synth_seq = overnight_synth_flat.view(n_real, MAX_AUDIO_TIME_STEPS, N_MFCC)

    sampled_idx = torch.randint(low=0, high=real_train_labels.size(0), size=(n_real,))
    overnight_synth_labels = real_train_labels[sampled_idx].clone()

    real_seq = real_flat_features.view(n_real, MAX_AUDIO_TIME_STEPS, N_MFCC)
    aug_x = torch.cat([real_seq, overnight_synth_seq], dim=0)
    aug_y = torch.cat([real_train_labels.clone(), overnight_synth_labels], dim=0)
    overnight_aug_loader = DataLoader(TensorDataset(aug_x, aug_y), batch_size=AUDIO_BATCH_SIZE, shuffle=True)

    # 4) Train Net6 full
    print("[Overnight] Step 4/6: Training Net6 full on augmented data...")
    set_seed(SEED)
    overnight_net6 = Net6LSTM()
    overnight_net6, overnight_net6_hist = train_audio_model(
        model=overnight_net6,
        model_name="Net6",
        epochs=AUDIO_FULL_EPOCHS,
        train_loader=overnight_aug_loader,
        val_loader=val_audio_loader,
        lr=AUDIO_LEARNING_RATE,
    )
    overnight_net6_test_loss, overnight_net6_test_acc = test_audio_model(overnight_net6, test_audio_loader)
    append_audio_result("Net6_overnight", "Adam", AUDIO_FULL_EPOCHS, overnight_net6_hist, overnight_net6_test_loss, overnight_net6_test_acc)

    # 5) Save results CSV
    print("[Overnight] Step 5/6: Saving results_audio_models.csv...")
    overnight_df = pd.DataFrame(audio_results_records)
    overnight_df.to_csv("results_audio_models.csv", index=False)

    # 6) Final summary
    print("[Overnight] Step 6/6: Final summary table")
    if overnight_df.empty:
        print("[Overnight] No results logged.")
    else:
        display(overnight_df.sort_values(["model_name", "epochs"]).reset_index(drop=True))

    print("[Overnight] Pipeline complete.")
else:
    print("Overnight run skipped. Set RUN_OVERNIGHT_NET5_NET6=True when ready.")
